In [80]:
# import libraries

import pandas  as pd
import numpy as np
import matplotlib.pyplot as mp
import seaborn as sb 
import csv 


In [81]:
df = pd.read_csv('cleaned_retail.csv',dtype={'Invoice': str})


In [82]:
df.head()


,Unnamed: 0,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue
0,0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4
1,1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
2,2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
3,3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8
4,4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0


In [83]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1041670 entries, 0 to 1041669
Data columns (total 10 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Unnamed: 0   1041670 non-null  int64  
 1   Invoice      1041670 non-null  str    
 2   StockCode    1041670 non-null  str    
 3   Description  1041670 non-null  str    
 4   Quantity     1041670 non-null  int64  
 5   InvoiceDate  1041670 non-null  str    
 6   Price        1041670 non-null  float64
 7   Customer ID  805549 non-null   float64
 8   Country      1041670 non-null  str    
 9   Revenue      1041670 non-null  float64
dtypes: float64(3), int64(2), str(5)
memory usage: 79.5 MB


In [84]:
# Drop col 
df = df.drop(columns=['Unnamed: 0'])


In [85]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0


In [86]:
# Extract Date 
df['InvoiceDate']=pd.to_datetime(df['InvoiceDate'])
df['Date']=df['InvoiceDate'].dt.date
df.head()


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue,Date
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4,2009-12-01
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0,2009-12-01
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0,2009-12-01
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8,2009-12-01
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0,2009-12-01


In [87]:
# Total quantity sold per SKU per day
daily_demand = df.groupby(['StockCode','Description','Date'])['Quantity'].sum().reset_index()
daily_demand.rename(columns={'Quantity':'DailyQty'},inplace='True')
print(daily_demand.shape)
print(daily_demand.head(10))
daily_demand.to_csv("daily_demand.csv",index=False)

(534654, 4)
  StockCode                  Description        Date  DailyQty
0     10002  INFLATABLE POLITICAL GLOBE   2009-12-01        12
1     10002  INFLATABLE POLITICAL GLOBE   2009-12-03         7
2     10002  INFLATABLE POLITICAL GLOBE   2009-12-04        73
3     10002  INFLATABLE POLITICAL GLOBE   2009-12-06        49
4     10002  INFLATABLE POLITICAL GLOBE   2009-12-07         2
5     10002  INFLATABLE POLITICAL GLOBE   2009-12-08        12
6     10002  INFLATABLE POLITICAL GLOBE   2009-12-10         1
7     10002  INFLATABLE POLITICAL GLOBE   2009-12-11         9
8     10002  INFLATABLE POLITICAL GLOBE   2009-12-14        37
9     10002  INFLATABLE POLITICAL GLOBE   2009-12-21        12


In [88]:
df_2 = pd.read_csv('daily_demand.csv',dtype={'StockCode': str})
df_2.head()
df_2.info()

<class 'pandas.DataFrame'>
RangeIndex: 534654 entries, 0 to 534653
Data columns (total 4 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   StockCode    534654 non-null  str  
 1   Description  534654 non-null  str  
 2   Date         534654 non-null  str  
 3   DailyQty     534654 non-null  int64
dtypes: int64(1), str(3)
memory usage: 16.3 MB


In [89]:
# How many days of dataset
df['Date']=pd.to_datetime(df['Date'])

full_date_range=pd.date_range(df_2['Date'].min(),df_2['Date'].max())
total_days = len(full_date_range)
print(f"Total days in dataset : {total_days}")


Total days in dataset : 739


In [90]:
# For each SKU , total quantity sold and count of days  it appeared
sku_stats=df_2.groupby(['StockCode','Description']).agg(
    TotalQtySold =('DailyQty','sum'),
    DaysWithSales=('DailyQty','count'),
    StdDevOnSaleDays=('DailyQty','std')
).reset_index()

print(sku_stats)


         StockCode                         Description  TotalQtySold  \
0            10002         INFLATABLE POLITICAL GLOBE           8836   
1           10002R               ROBOT PENCIL SHARPNER             4   
2            10080            GROOVY CACTUS INFLATABLE           315   
3            10109                BENDY COLOUR PENCILS             4   
4            10120                        DOGGY RUBBER           680   
...            ...                                 ...           ...   
5626  gift_0001_40  Dotcomgiftshop Gift Voucher £40.00             5   
5627  gift_0001_50  Dotcomgiftshop Gift Voucher £50.00             6   
5628  gift_0001_70  Dotcomgiftshop Gift Voucher £70.00             1   
5629  gift_0001_80  Dotcomgiftshop Gift Voucher £80.00             1   
5630             m                              Manual             5   

      DaysWithSales  StdDevOnSaleDays  
0               225         93.088631  
1                 3          0.577350  
2              

In [91]:
# Average daily demand 
sku_stats['AvgDailyDemand']=sku_stats['TotalQtySold'] / total_days
sku_stats['AvgDailyDemand_ActiveDays'] = sku_stats['TotalQtySold'] / sku_stats['DaysWithSales']

In [92]:

sku_stats['StdDevOnSaleDays']=sku_stats['StdDevOnSaleDays'].fillna(0)

In [93]:
# print(sku_stats)
print(sku_stats.shape)
print(sku_stats.sort_values(by='TotalQtySold',ascending=False).head(10))
sku_stats.to_csv('sku_stats.csv',index=False)

(5631, 7)
     StockCode                         Description  TotalQtySold  \
4171     84077   WORLD WAR 2 GLIDERS ASSTD DESIGNS        110138   
5000    85123A  WHITE HANGING HEART T-LIGHT HOLDER         96080   
4640     84879       ASSORTED COLOUR BIRD ORNAMENT         81809   
3355     23843         PAPER CRAFT , LITTLE BIRDIE         80995   
4967    85099B             JUMBO BAG RED RETROSPOT         78860   
2810     23166      MEDIUM CERAMIC TOP STORAGE JAR         78033   
121      17003                 BROCADE RING PURSE          71430   
1384     21977  PACK OF 60 PINK PAISLEY CAKE CASES         56794   
4801     84991         60 TEATIME FAIRY CAKE CASES         54716   
1607     22197                SMALL POPCORN HOLDER         49948   

      DaysWithSales  StdDevOnSaleDays  AvgDailyDemand  \
4171            470        527.825601      149.036536   
5000            602        218.784961      130.013532   
4640            582        243.664243      110.702300   
3355         

In [94]:
# Look at the raw data for this specific product to see why it has 80k+ quantity in 1 day
df_2[df_2['StockCode'] == '23843']
sku_stats[sku_stats['StockCode'] == '23843']
#You can leave it in your data, but during your presentation, you must point it out and say: "While 'Paper Craft, Little Birdie' appears to be a top product by volume, it is an extreme outlier driven entirely by a single wholesale transaction of 80,995 units on a single day."

,StockCode,Description,TotalQtySold,DaysWithSales,StdDevOnSaleDays,AvgDailyDemand,AvgDailyDemand_ActiveDays
3355,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,1,0.0,109.600812,80995.0


In [95]:
df_3 =  pd.read_csv("sku_stats.csv")
df_3.info()

<class 'pandas.DataFrame'>
RangeIndex: 5631 entries, 0 to 5630
Data columns (total 7 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   StockCode                  5631 non-null   str    
 1   Description                5631 non-null   str    
 2   TotalQtySold               5631 non-null   int64  
 3   DaysWithSales              5631 non-null   int64  
 4   StdDevOnSaleDays           5631 non-null   float64
 5   AvgDailyDemand             5631 non-null   float64
 6   AvgDailyDemand_ActiveDays  5631 non-null   float64
dtypes: float64(3), int64(2), str(2)
memory usage: 308.1 KB


In [96]:
lead_time_days = 7
service_level_z = 1.65
# Formula : Safety Stock = Z * StdDev(demand) * sqrt(Lead Time)
df_3['Safety_Stock'] = service_level_z * df_3['StdDevOnSaleDays'] * np.sqrt(lead_time_days)
df_3['Safety_Stock']=df_3['Safety_Stock'].round(0)

In [97]:
# formula : reorder point = ( avg daily demand * lead time) + safety stock
df_3['ReorderPoint'] = (df_3['AvgDailyDemand'] * lead_time_days) + df_3['Safety_Stock']
df_3['ReorderPoint']=df_3['ReorderPoint'].round(0)
print(df_3.sort_values(by='TotalQtySold', ascending=False).head(10))
df_3.to_csv("sku_reorder_point.csv",index=False)

     StockCode                         Description  TotalQtySold  \
4171     84077   WORLD WAR 2 GLIDERS ASSTD DESIGNS        110138   
5000    85123A  WHITE HANGING HEART T-LIGHT HOLDER         96080   
4640     84879       ASSORTED COLOUR BIRD ORNAMENT         81809   
3355     23843         PAPER CRAFT , LITTLE BIRDIE         80995   
4967    85099B             JUMBO BAG RED RETROSPOT         78860   
2810     23166      MEDIUM CERAMIC TOP STORAGE JAR         78033   
121      17003                 BROCADE RING PURSE          71430   
1384     21977  PACK OF 60 PINK PAISLEY CAKE CASES         56794   
4801     84991         60 TEATIME FAIRY CAKE CASES         54716   
1607     22197                SMALL POPCORN HOLDER         49948   

      DaysWithSales  StdDevOnSaleDays  AvgDailyDemand  \
4171            470        527.825601      149.036536   
5000            602        218.784961      130.013532   
4640            582        243.664243      110.702300   
3355              1    

In [98]:
df_4=pd.read_csv('sku_reorder_point.csv')
df_4.info()

<class 'pandas.DataFrame'>
RangeIndex: 5631 entries, 0 to 5630
Data columns (total 9 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   StockCode                  5631 non-null   str    
 1   Description                5631 non-null   str    
 2   TotalQtySold               5631 non-null   int64  
 3   DaysWithSales              5631 non-null   int64  
 4   StdDevOnSaleDays           5631 non-null   float64
 5   AvgDailyDemand             5631 non-null   float64
 6   AvgDailyDemand_ActiveDays  5631 non-null   float64
 7   Safety_Stock               5631 non-null   float64
 8   ReorderPoint               5631 non-null   float64
dtypes: float64(5), int64(2), str(2)
memory usage: 396.1 KB


#This dataset only has historical sales — it doesn't tell us what's sitting in the warehouse today. Real companies pull this from their inventory system, so here we simulate it realistically and document it clearly as an assumption.

In [101]:
# Asssumptions based Current Stock
np.random.seed(42)
df_4['SimulatedDaysOfStock']=np.random.randint(1,20,size=len(df_4))
df_4['CurrentStock'] = df_4['AvgDailyDemand'] * df_4['SimulatedDaysOfStock']

In [107]:
# Reorder flag 
df_4['ReorderFlag'] = np.where(df_4['CurrentStock'] < df_4['ReorderPoint'],'Reorder Now','OK')
print(df_4['ReorderFlag'].value_counts())
print(df_4.sort_values(by='TotalQtySold',ascending=False).head())
df_4.to_csv('final_inventory_analysis.csv',index=False)

ReorderFlag
Reorder Now    5200
OK              431
Name: count, dtype: int64
     StockCode                         Description  TotalQtySold  \
4171     84077   WORLD WAR 2 GLIDERS ASSTD DESIGNS        110138   
5000    85123A  WHITE HANGING HEART T-LIGHT HOLDER         96080   
4640     84879       ASSORTED COLOUR BIRD ORNAMENT         81809   
3355     23843         PAPER CRAFT , LITTLE BIRDIE         80995   
4967    85099B             JUMBO BAG RED RETROSPOT         78860   

      DaysWithSales  StdDevOnSaleDays  AvgDailyDemand  \
4171            470        527.825601      149.036536   
5000            602        218.784961      130.013532   
4640            582        243.664243      110.702300   
3355              1          0.000000      109.600812   
4967            461        196.818373      106.711773   

      AvgDailyDemand_ActiveDays  Safety_Stock  ReorderPoint  \
4171                 234.336170        2304.0        3347.0   
5000                 159.601329         955.

In [110]:
df_5 = pd.read_csv('final_inventory_analysis.csv')
df_5.info()

<class 'pandas.DataFrame'>
RangeIndex: 5631 entries, 0 to 5630
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   StockCode                  5631 non-null   str    
 1   Description                5631 non-null   str    
 2   TotalQtySold               5631 non-null   int64  
 3   DaysWithSales              5631 non-null   int64  
 4   StdDevOnSaleDays           5631 non-null   float64
 5   AvgDailyDemand             5631 non-null   float64
 6   AvgDailyDemand_ActiveDays  5631 non-null   float64
 7   Safety_Stock               5631 non-null   float64
 8   ReorderPoint               5631 non-null   float64
 9   SimulatedDaysOfStock       5631 non-null   int64  
 10  CurrentStock               5631 non-null   float64
 11  ReorderFlag                5631 non-null   str    
dtypes: float64(6), int64(3), str(3)
memory usage: 528.0 KB


In [117]:
# Total revenue per SKU
transactions = pd.read_csv('cleaned_retail.csv',dtype={'Invoice' : str})
sku_revenue=transactions.groupby('StockCode')['Revenue'].sum().reset_index()
sku_revenue.rename(columns={'Revenue':'TotalRevenue'},inplace=True)


In [126]:
# Merge revenue with main dataframe
df_5=df_5.merge(sku_revenue,on='StockCode',how='left')
df_5=df_5.sort_values(by='TotalRevenue',ascending=False).reset_index(drop=True)


In [129]:
# df_5.drop(columns=['TotalRevenue_y','TotalRevenue_x'],inplace=True)

# Cumulative % of Total Revenue
df_5['CumulativeRevenue'] = df_5['TotalRevenue'].cumsum()
df_5['CumulativeRevenuePercent']=df_5['CumulativeRevenue'] / df_5['TotalRevenue'].sum() * 100



In [ ]:
# Assign ABC category based on Total Revenue / Pareto Analysis 80/20 rule 
def assign_abc(percent):
    if percent <=80:
        return 'A'
    elif percent <=95 :
        return 'B'
    else :
        return 'C' 

df_5['ABC_Category']=df_5['CumulativeRevenuePercent'].apply(assign_abc)

In [134]:
# Summary
pd.options.display.float_format = '{:,.2f}'.format
print(df_5['ABC_Category'].value_counts())
print(df_5.groupby('ABC_Category')['TotalRevenue'].sum())
df_5.to_csv("final_inventory_with_abc.csv", index=False)

ABC_Category
C    2972
B    1488
A    1171
Name: count, dtype: int64
ABC_Category
A   22,050,424.82
B    4,135,458.18
C    1,378,803.06
Name: TotalRevenue, dtype: float64
